# massive in silico screening with Prophet

This notebook demonstrates how to make predictions with Prophet with any of the checkpoints we have made available.

In [2]:
import sys
import os 
import torch
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.append("../")
sys.path.append("../prophet")
import pandas as pd
import yaml
from Prophet import Prophet
from config import set_config
from dataloader import _choose, universal_processing


We load in a config file to automatically get the file paths for the embeddings which were used, but any of the files in `embeddings` can be passed.

In [3]:
with open('config_file_finetuning.yaml', 'r') as f:
    config = set_config(yaml.safe_load(f))

In [4]:
# replace this with the path to the model checkpoint you want to use
path = '/lustre/groups/ml01/projects/super_rad_project/pretrained_prophet/GDSC/cl_0_TrainedOn545_cl_out_300cl_1219iv_512model_8layers_Falsesimpler_Truemask_0.0001lr_Falseexplicitphenotype_5000warmup_40000max_iters_Falseunbalanced_0.01wd_256bs_Trueft/cl_0_TrainedOn545_seed_110/epoch=19.ckpt'

In [5]:
print(torch.cuda.is_initialized())

False


In [6]:
model = Prophet(
    iv_emb_path=config.genes_prior,
    cl_emb_path=config.cell_lines_prior,
    ph_emb_path=None,
    model_pth=config.ckpt_path,
)

returning trained model!
Gene net:  Sequential(
  (0): Linear(in_features=1219, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Cell line net:  Sequential(
  (0): Linear(in_features=300, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Regressor:  Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
  (4): GELU(approximate='none')
  (5): Linear(in_features=512, out_features=1, bias=True)
)


In [7]:
print(torch.cuda.is_initialized())

False


In [8]:
gdsc_data_path = "/lustre/groups/ml01/projects/super_rad_project/data/GDSC_notscaled_minmax.csv" #/lustre/groups/ml01/projects/super_rad_project/data/JUMP_all_scaled.csv" #"/lustre/groups/ml01/projects/super_rad_project/data/GDSC_scaled.csv"
data_label = pd.read_csv(gdsc_data_path, index_col = 0)

data_label.insert(2, "iv2", "negative_drug")
data_label.insert(3, "phenotype", "GDSC")

data_label = universal_processing(data_label)
data_label


,cell_line,iv1,iv2,phenotype,value,iv_name
0,PFSK1,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,negative_drug,GDSC,0.323078,Camptothecin
1,A673,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,negative_drug,GDSC,0.172422,Camptothecin
2,ES5,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,negative_drug,GDSC,0.239133,Camptothecin
3,ES7,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,negative_drug,GDSC,0.164659,Camptothecin
4,EW11,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,negative_drug,GDSC,0.222290,Camptothecin
...,...,...,...,...,...,...
484067,SNU175,negative_drug,cc(=o)nc(cs)c(=o)o,GDSC,0.835833,N-acetyl cysteine
484068,SNU407,negative_drug,cc(=o)nc(cs)c(=o)o,GDSC,0.766903,N-acetyl cysteine
484069,SNU61,negative_drug,cc(=o)nc(cs)c(=o)o,GDSC,0.852908,N-acetyl cysteine
484070,SNUC5,negative_drug,cc(=o)nc(cs)c(=o)o,GDSC,0.860900,N-acetyl cysteine


In [9]:
model.train(
    data_label,
    iv_col = ['iv1', 'iv2'],
    cl_col = 'cell_line',
    ph_col = 'phenotype',
    model_config = config
)

Removing 61 such as ['123138', '123829', '150412', '50869', '615590'] from ['iv1', 'iv2']. 393646 rows remaining.
Removing 285 such as ['451LU', '7860', 'ALLPO', 'ARH77', 'ATN1'] from ['cell_line']. 280202 rows remaining.
Fitting model.
pytorch model, finetuning
df.index: RangeIndex(start=0, stop=280202, step=1)


: 

Suppose we have some small molecules, some cell lines we would like to test them in, and we're interested in measuring their relative IC50. We can pass in lists of these inputs, and Prophet will return predictions for all combinations:

#### Making predictions by passing all treatments, cell lines, and phenotypes you want to run

This format can be useful when running large combinatorial screens in silico, as it splits the experiments up into batches to help prevent memory errors.

In [ ]:
iv_list = [
    'oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc4=c(c=c(c=c4)i)f)=o',
    'cc(nc1=cc=cc(n(c2=o)c(c(c(n2c3cc3)=o)=c(n4c)nc5=cc=c(c=c5f)i)=c(c4=o)c)=c1)=o',
    'fc1=cc=c(c(f)=c1c(c2=cnc3=nc=c(c=c32)c4=cc=c(c=c4)cl)=o)ns(ccc)(=o)=o',
    'cs(=o)c'  # DMSO
]
cl_list = ['A375','UACC62','WM983B','MALME3M','A2058','WM793','HT144','RPMI7951','WM1799','LOXIMVI','WM2664','WM88','G361','SKMEL24','WM115', 'SKMEL2', 'SKMEL1', 'HMCB', 'MDAMB435S', 'UACC257']
ph_list = ['GDSC']

In [ ]:
# predict with lists of treatments and cell lines
df = model.predict(
    target_ivs=iv_list,
    target_cls=cl_list,
    target_phs=ph_list,
    iv_col=['iv1', 'iv2'],  # pass to turn on combinatorial predictions
    num_iterations=1, save=False,
)
df

There are 1 iterations


  0%|          | 0/1 [00:00<?, ?it/s]

Removing 0 such as [] from ['iv1', 'iv2']. 200 rows remaining.
Removing 0 such as [] from ['cell_line']. 200 rows remaining.


/home/icb/ahmet.kaya/miniconda3/envs/prophet_api/lib/python3.12/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/ahmet.kaya/.local/lib/python3.12/site-pack ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/icb/ahmet.kaya/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:67: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install

Predicting DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

/home/icb/ahmet.kaya/miniconda3/envs/prophet_api/lib/python3.12/site-packages/torch/nn/modules/transformer.py:408: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


Predicting DataLoader 0: 100%|██████████| 1/1 [00:05<00:00,  0.20it/s]


100%|██████████| 1/1 [01:59<00:00, 119.83s/it]


,iv1,iv2,cell_line,phenotype,iv1+iv2,value,pred
0,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A375,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.186035
1,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,UACC62,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.158411
2,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM983B,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.312883
3,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,MALME3M,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.261691
4,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A2058,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.395144
...,...,...,...,...,...,...,...
252,cs(=o)c,cs(=o)c,SKMEL2,GDSC,cs(=o)c+cs(=o)c,_,0.591978
253,cs(=o)c,cs(=o)c,SKMEL1,GDSCcomb,cs(=o)c+cs(=o)c,_,0.620441
254,cs(=o)c,cs(=o)c,HMCB,PRISM,cs(=o)c+cs(=o)c,_,0.638921
255,cs(=o)c,cs(=o)c,MDAMB435S,inhouse,cs(=o)c+cs(=o)c,_,0.577903


#### Making predictions for a specific set of treatments, cell lines, and phenotypes

If we're interested in only a subset of the experimental matrix, we can also pass in a custom dataframe. (This is the recommended usage, as users understand exactly the list being predicted.)

In [ ]:
# construct a dataframe containing the experiments we want to run
experiments_df = pd.MultiIndex.from_product([
    iv_list,
    cl_list,
], names=['iv1', 'cell_line'])
experiments_df = experiments_df.to_frame(index=False).reset_index(drop=True)
experiments_df

,iv1,cell_line
0,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A375
1,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,UACC62
2,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM983B
3,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,MALME3M
4,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A2058
...,...,...
75,cs(=o)c,SKMEL2
76,cs(=o)c,SKMEL1
77,cs(=o)c,HMCB
78,cs(=o)c,MDAMB435S


In [ ]:
input_df = experiments_df.copy()
input_df['iv2'] = 'cs(=o)c'  # DMSO
input_df['phenotype'] = 'GDSC'
df = model.predict(input_df, num_iterations=1, save=False)
df

There are 1 iterations


  0%|          | 0/1 [00:00<?, ?it/s]/home/icb/ahmet.kaya/miniconda3/envs/prophet_api/lib/python3.12/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/ahmet.kaya/.local/lib/python3.12/site-pack ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  1.65it/s]


100%|██████████| 1/1 [00:20<00:00, 20.95s/it]


,iv1,cell_line,iv2,phenotype,pred
0,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A375,cs(=o)c,GDSC,0.374227
1,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,UACC62,cs(=o)c,GDSC,0.299597
2,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM983B,cs(=o)c,GDSC,0.431186
3,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,MALME3M,cs(=o)c,GDSC,0.391419
4,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A2058,cs(=o)c,GDSC,0.505639
...,...,...,...,...,...
132,cs(=o)c,SKMEL2,cs(=o)c,GDSC,0.591978
133,cs(=o)c,SKMEL1,cs(=o)c,GDSCcomb,0.620440
134,cs(=o)c,HMCB,cs(=o)c,PRISM,0.638921
135,cs(=o)c,MDAMB435S,cs(=o)c,inhouse,0.577903
